# Классификатор тональности русских твитов: TF-IDF + MLP

**Задача:** бинарная классификация твитов на позитивные и негативные на датасете RuTweetCorp.

**Подход:** классический NLP-пайплайн: очистка и лемматизация текста → векторизация TF-IDF → многослойный перцептрон (MLP).
Дополнительно разбирается, на какие слова опирается модель, и где у такого подхода заканчивается «понимание» текста.

**Содержание**
1. Загрузка и предобработка данных
2. Векторизация TF-IDF и обучение MLP
3. Интерпретация модели: анализ весов
4. Оценка качества и слабые места
5. Тот же MLP на PyTorch
6. Отчёт и выводы

**Запуск:** Google Colab (`Runtime → Run all`) или локально через Docker (`docker compose up -d`, см. README).

## 0. Подготовка окружения

В Colab большинство библиотек уже установлено, не хватает только морфологического анализатора `pymorphy3` для лемматизации.
В Docker-образе всё ставится из `requirements.txt`, поэтому установка выполняется только в Colab.

In [1]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q pymorphy3 pymorphy3-dicts-ru

Фиксируем `seed` (зерно генератора случайных чисел): разбиение на выборки, перемешивание и начальные веса нейросети
будут одинаковыми при каждом запуске, и результаты можно воспроизвести.

In [2]:
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

pd.set_option('display.max_colwidth', 120)

## 1. Загрузка и предобработка данных

### 1.1 Загрузка датасета

[RuTweetCorp на Kaggle](https://www.kaggle.com/datasets/maximsuvorov/rutweetcorp) содержит два размеченных файла:
`positive.csv` и `negative.csv`. Текст твита лежит в колонке `ttext`.

Файлы скачиваются по прямой ссылке без аккаунта Kaggle и сохраняются в папку `data/`. При повторном запуске берутся с диска.

Метки классов: **позитив = 1**, **негатив = 0**. Модели бинарной классификации ожидают классы 0 и 1,
а `predict_proba` возвращает вероятность класса 1, то есть «вероятность позитива».

In [3]:
DATA_DIR = Path('../data') if Path('../data').is_dir() else Path('data')
DATA_DIR.mkdir(exist_ok=True)

KAGGLE_URL = 'https://www.kaggle.com/api/v1/datasets/download/maximsuvorov/rutweetcorp/{name}'

DATASET_FILES = {'positive.csv': 1, 'negative.csv': 0}
LABEL_NAMES = {0: 'негатив', 1: 'позитив'}


def download_dataset_file(name: str) -> Path:
    csv_path = DATA_DIR / name
    if csv_path.exists():
        print(f'{name}: уже скачан')
        return csv_path

    print(f'{name}: скачиваю...')
    response = requests.get(KAGGLE_URL.format(name=name), timeout=300)
    response.raise_for_status()

    zip_path = DATA_DIR / f'{name}.zip'
    zip_path.write_bytes(response.content)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extract(name, DATA_DIR)
    zip_path.unlink()

    return csv_path


frames = []
for name, label in DATASET_FILES.items():
    part = pd.read_csv(download_dataset_file(name), usecols=['ttext'])
    part['label'] = label
    frames.append(part)

df = (
    pd.concat(frames, ignore_index=True)
    .rename(columns={'ttext': 'text'})
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print(f'Размер датасета: {df.shape[0]:,} твитов, {df.shape[1]} колонки')
df.head(10)

positive.csv: уже скачан


negative.csv: уже скачан


Размер датасета: 226,834 твитов, 2 колонки


,text,label
0,мыс на меня обиделась:(\r\nя ей даже ничего не сделала:(,0
1,"аааааааааааааааааааа,не хочу на работу :(",0
2,"У меня какой-то особенный вид ушей! :D, некоторые вакуумные наушники в моих ушах просто не держатся!",1
3,"@simonovkon он неплохой человек в жизни. Я работала в шоу-бизе, со многими знакома. Встречаются очень хорошие люди....",0
4,"RT @Darina_Lo: Домааааа\nЕхали на такси, пели песни, отдыхали.\nКричали на улице:)\nМы настоящяя семья)",1
5,@passkrinDObroTA это ужасно:( так страашно,0
6,"RT @miwisakovab: Не грустим за летом, скоро будет следующее)",1
7,"Лежим с молой в постели , помогаем ему выбирать машину !)",1
8,"Я ВАС НАЙДУ, ОТРЕЖУ ПАЛЬЦЫ И ЗАСТАВЛЮ СОЖРАТЬ, ЕСЛИ ЕЩЕ РАЗ ДОБАВИТЕ МЕНЯ В ОБЩУЮ БЕСЕДУ ВКОНТАКТЕ С КАКОЙ-ТО ЛЕВОЙ ...",1
9,"@larkagerasimova Контоев Артем) рост, вес и тд интересует ?)",1


### 1.2 Первичный осмотр данных

Проверяем баланс классов, пропуски, дубликаты и длину твитов.

In [4]:
balance = pd.DataFrame({
    'твитов': df['label'].value_counts(),
    'доля': df['label'].value_counts(normalize=True).round(3),
}).rename(index=LABEL_NAMES)

print('Пустых текстов:', df['text'].isna().sum())
print('Дубликатов текста:', df['text'].duplicated().sum())
balance

Пустых текстов: 0
Дубликатов текста: 9394


,твитов,доля
label,,
позитив,114911,0.507
негатив,111923,0.493


In [5]:
words = df['text'].str.split().str.len()

words.groupby(df['label'].map(LABEL_NAMES)).describe().round(1)

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
негатив,111923.0,12.0,5.0,1.0,8.0,11.0,15.0,37.0
позитив,114911.0,12.5,4.7,1.0,9.0,12.0,16.0,40.0


### 1.3 Как размечен корпус: смайлики

Корпус размечали автоматически: твиты со «весёлыми» смайликами (`:)`, `)))`, `:D`) попали в позитив,
с «грустными» (`:(`, `(((`) - в негатив. Смайлики остались в текстах.

Посчитаем, в какой доле твитов каждого класса есть скобки:

In [6]:
brackets = pd.DataFrame({
    'есть ")"': df['text'].str.contains(')', regex=False),
    'есть "("': df['text'].str.contains('(', regex=False),
}).groupby(df['label'].map(LABEL_NAMES)).mean().round(3)

brackets

,"есть "")""","есть ""("""
label,,
негатив,0.000,0.947
позитив,0.829,0.000


**Что видно из таблицы:**
- в позитивных твитах `)` встречается в 82.9% случаев, а `(` не встречается ни разу;
- в негативных твитах `(` встречается в 94.7% случаев, а `)` не встречается ни разу.

Скобка почти однозначно выдаёт класс. Если её оставить, модель выучит правило «`)` → позитив, `(` → негатив»,
покажет высокую точность и при этом ничего не узнает о словах. Это **утечка данных (data leakage)**:
признак, по которому ставили метку, попал во входные данные. Поэтому при очистке текста (раздел 1.4)
скобки и смайлики удаляются обязательно.

**Другие наблюдения:**
- классы почти сбалансированы (50.7% / 49.3%), поэтому Accuracy будет честной метрикой;
- в среднем 12 слов на твит, длина у классов почти одинаковая;
- 9 394 текста повторяются (в основном ретвиты), с ними разберёмся после очистки;
- разметка шумная: она отражает смайлик, а не смысл. Например, угроза капслоком в строке 8 размечена как позитив
  (в ней был смайлик), а «он неплохой человек» в строке 3 — как негатив.